In [ ]:
import os
import re
import json
import shutil
import hashlib
import subprocess
import requests
import urllib.parse
from datetime import datetime, timedelta
from typing import TypedDict, List, Dict, Any
import pandas as pd
from bs4 import BeautifulSoup

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from exa_py import Exa
from composio import Composio
from dotenv import load_dotenv
load_dotenv()  # This loads your variables from the local .env file

# ==========================================
# 1. API KEYS & CREDENTIALS SETUP
# ==========================================
os.environ ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
os.environ ["COMPOSIO_API_KEY"] = os.getenv("COMPOSIO_API_KEY", "")
os.environ ["GETEMAIL_API_KEY"] = os.getenv("GETEMAIL_API_KEY", "")

# Initialize Search Clients
composio = Composio(api_key=os.environ["COMPOSIO_API_KEY"])
exa = Exa(api_key=os.environ["EXA_API_KEY"])

# Local LLM for Private CV Ingestion & Fallback
ollama_llm = ChatOllama(model="qwen2.5:7b", base_url="http://localhost:11434")
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

# Directories & Storage
RESUMES_DIR = r"C:\Users\simar\Desktop\CV_2026"
CACHE_FILE = r"C:\Users\simar\Downloads\multi_resume_cache.json"
CHROMA_PERSIST_DIR = r"C:\Users\simar\Downloads\chroma_db_resumes"
EXCEL_PATH = r"C:\Users\simar\Downloads\Matched_Jobs.xlsx"

# Path to authenticated Claude executable
CLAUDE_EXE = r"C:\Users\simar\.local\bin\claude.exe"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_file_hash(filepath: str) -> str:
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

def load_cache() -> Dict[str, Any]:
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache_data: Dict[str, Any]):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache_data, f, indent=2)

def parse_duration_to_days(user_input: str) -> int:
    txt = user_input.lower().strip()
    if "24" in txt or "hour" in txt or "today" in txt or "1 day" in txt:
        return 1
    day_match = re.search(r"(\d+)\s*day", txt)
    if day_match:
        return int(day_match.group(1))
    if "week" in txt:
        week_match = re.search(r"(\d+)\s*week", txt)
        return int(week_match.group(1)) * 7 if week_match else 7
    if "month" in txt:
        month_match = re.search(r"(\d+)\s*month", txt)
        return int(month_match.group(1)) * 30 if month_match else 30
    return 7

# ==========================================
# 3. CHROMA SETUP
# ==========================================
vectorstore = Chroma(
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=ollama_embeddings
)

def sync_resumes_to_chroma(resumes_dir: str):
    if not os.path.exists(resumes_dir):
        return
    cache = load_cache()
    pdf_files = [f for f in os.listdir(resumes_dir) if f.endswith(".pdf")]

    for file_name in pdf_files:
        file_path = os.path.join(resumes_dir, file_name)
        file_hash = get_file_hash(file_path)
        persona_key = os.path.splitext(file_name)[0]

        if persona_key in cache and cache[persona_key].get("pdf_hash") == file_hash:
            continue

        print(f"📦 Indexing updated resume persona: {file_name} into ChromaDB...")
        loader = PyPDFLoader(file_path)
        docs = loader.load_and_split()
        for d in docs:
            d.metadata["persona_key"] = persona_key
            d.metadata["file_name"] = file_name

        vectorstore.add_documents(docs)
        print(f"  ✓ {file_name} indexed with persona metadata.")

sync_resumes_to_chroma(RESUMES_DIR)

# ==========================================
# 4. PIPELINE STATE
# ==========================================
class PipelineState(TypedDict):
    candidate_name: str
    roles: List[str]
    target_geos: List[str]
    job_duration_days: int
    duration_label: str
    target_approved_count: int
    resume_summary: str
    raw_jobs: List[Dict[str, Any]]
    evaluated_jobs: List[Dict[str, Any]]
    approved_jobs: List[Dict[str, Any]]
    low_match_summary: str
    search_iteration: int
    user_decision: str

# ==========================================
# 5. GRAPH NODES
# ==========================================

# Node 1: Multi-Resume Cached Reader
def parse_resume_node(state: PipelineState) -> Dict[str, Any]:
    persona_key = state.get("candidate_name", "Default")
    print(f"\n📄 Node 1: Checking Resume Profile Cache for '{persona_key}'...")

    cache = load_cache()
    expected_file = os.path.join(RESUMES_DIR, f"{persona_key}.pdf")
    current_hash = get_file_hash(expected_file) if os.path.exists(expected_file) else None

    if persona_key in cache:
        entry = cache[persona_key]
        if current_hash is None or entry.get("pdf_hash") == current_hash:
            print(f"  ⚡ [Instant] Loaded cached profile for '{persona_key}' (0ms)!")
            return {"resume_summary": entry["summary"]}

    print(f"  ⚙️ Generating new summary for '{persona_key}' via local Ollama...")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"persona_key": persona_key}})
    docs = retriever.invoke(f"Key technical skills, target roles, impact for {persona_key}")
    context = "\n\n".join([d.page_content for d in docs])

    if not context.strip() and os.path.exists(expected_file):
        loader = PyPDFLoader(expected_file)
        context = "\n\n".join([d.page_content for d in loader.load()])

    prompt = f"""Summarize this candidate profile into concise bullet points for job matching:
1. Primary Roles (e.g. Chief of Staff, Product Manager, AI Product Builder)
2. Core Technical, AI & Product Competencies
3. Industry Domains & Impact

Resume Context:
{context}"""

    resp = ollama_llm.invoke(prompt)
    summary_text = resp.content

    cache[persona_key] = {
        "file_name": f"{persona_key}.pdf",
        "pdf_hash": current_hash,
        "last_updated": datetime.now().isoformat(),
        "summary": summary_text
    }
    save_cache(cache)
    print(f"  ✓ Summary cached successfully for '{persona_key}'!")
    return {"resume_summary": summary_text}


# Fast Live LinkedIn Direct Guest Scraper
def fetch_live_linkedin_jobs(role: str, geo: str, days: int = 3, limit: int = 20) -> List[Dict[str, Any]]:
    seconds = days * 86400
    encoded_role = urllib.parse.quote(role)
    encoded_geo = urllib.parse.quote(geo)
    url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={encoded_role}&location={encoded_geo}&f_TPR=r{seconds}&start=0"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
    
    jobs = []
    try:
        resp = requests.get(url, headers=headers, timeout=8)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            for post in soup.find_all("li")[:limit]:
                title_tag = post.find("h3", class_="base-search-card__title")
                company_tag = post.find("h4", class_="base-search-card__subtitle")
                location_tag = post.find("span", class_="job-search-card__location")
                link_tag = post.find("a", class_="base-card__full-link")
                date_tag = post.find("time")
                
                if title_tag and link_tag:
                    jobs.append({
                        "Job Profile": title_tag.get_text(strip=True),
                        "Target Company": company_tag.get_text(strip=True) if company_tag else "Unknown Company",
                        "Location": location_tag.get_text(strip=True) if location_tag else geo,
                        "Job Link": link_tag.get("href", "").split("?")[0],
                        "Platform Name": "LinkedIn",
                        "Date Posted": date_tag.get_text(strip=True) if date_tag else f"Past {days} days"
                    })
    except Exception as e:
        print(f"  ⚠️ Live LinkedIn query issue for {role}: {e}")
    return jobs


# Node 2: Fast Dual-Engine Job Search
def search_jobs_node(state: PipelineState) -> Dict[str, Any]:
    days = state.get("job_duration_days", 3)
    iteration = state.get("search_iteration", 1)
    print(f"\n🔍 Node 2: Searching live postings [Iteration {iteration}] (past {days} days)...")

    all_jobs = []

    for role in state["roles"]:
        for geo in state["target_geos"]:
            print(f"  ⚡ Querying live LinkedIn index for '{role}' in '{geo}'...")
            li_jobs = fetch_live_linkedin_jobs(role, geo, days=days, limit=15)
            all_jobs.extend(li_jobs)

    for role in state["roles"]:
        for geo in state["target_geos"]:
            try:
                search_res = exa.search(
                    query=f'"{role}" {geo} hiring',
                    type="auto",
                    include_domains=["greenhouse.io", "lever.co", "ashbyhq.com"],
                    num_results=4
                )
                for r in search_res.results:
                    all_jobs.append({
                        "Job Profile": r.title.split("-")[0].strip(),
                        "Target Company": r.title.split("-")[-1].strip() if "-" in r.title else "Direct ATS",
                        "Location": geo,
                        "Job Link": r.url,
                        "Platform Name": "ATS (Greenhouse/Lever)",
                        "Date Posted": f"Past {days} days"
                    })
            except Exception:
                pass

    unique_map = {j["Job Link"]: j for j in all_jobs if j.get("Job Link")}
    final_raw = list(unique_map.values())

    print(f"  ✓ Found {len(final_raw)} fresh live postings in ~3 seconds!")
    return {"raw_jobs": final_raw, "search_iteration": iteration + 1}


# Direct Headless Claude Pro Evaluation (Windows-safe list args)
def call_claude_pro_evaluation(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    if not os.path.exists(CLAUDE_EXE):
        raise RuntimeError("Claude CLI executable not found at specified path.")

    prompt = f"""[CRITICAL INSTRUCTION: You are an automated batch evaluation script. Do NOT greet. Do NOT offer services. Output ONLY a valid JSON array and nothing else.]

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

Scoring Rules:
1. Product Manager, AI Product Builder, Strategy, or Chief of Staff roles that align with the candidate MUST score between 75 and 95.
2. Only flag pure software engineering (hands-on coding), ML model research/training, or low-level IT support as 'Low' (score < 50).
3. If a job fits product ownership, leadership, or business operations, assign 'Good' or 'Perfect'.

Return STRICTLY a JSON array matching this format:
[
  {{"id": 1, "match_score": 92, "rating": "Perfect", "rationale": "Direct fit for AI PM role.", "low_match_reason": null}},
  {{"id": 2, "match_score": 40, "rating": "Low", "rationale": "Requires ML research.", "low_match_reason": "Role demands ML systems engineering rather than PM leadership."}}
]
"""
    # List argument structure prevents cmd.exe from truncating multi-line strings
    res = subprocess.run(
        [CLAUDE_EXE, "-p", prompt, "--tools", ""],
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=90
    )
    if res.returncode == 0 and res.stdout.strip():
        match = re.search(r"\[.*\]", res.stdout, re.DOTALL)
        if match:
            eval_list = json.loads(match.group(0))
            return {item["id"]: item for item in eval_list}
    raise RuntimeError(f"Claude CLI execution issue: {res.stderr.strip()[:150]}")


# Working Local Qwen Evaluator Fallback
def evaluate_with_local_qwen(resume_summary: str, jobs_payload: str) -> Dict[int, Any]:
    print("  ⚙️ Invoking Local Qwen 2.5 (7B) to dynamically score all jobs...")
    prompt = f"""Compare this candidate resume profile against each job posting.

Candidate Profile:
{resume_summary}

Jobs to Evaluate:
{jobs_payload}

For each job, evaluate alignment with Product Management / Strategy:
- If the role is Product Manager, Product Lead, or Chief of Staff, assign match_score between 75 and 95.
- If it is pure software engineering or data science research, assign match_score < 50.

Return STRICTLY a JSON array:
[
  {{"id": 1, "match_score": 88, "rating": "Good", "rationale": "Strong fit for Product leadership.", "low_match_reason": null}}
]"""
    resp = ollama_llm.invoke(prompt)
    try:
        match = re.search(r"\[.*\]", resp.content, re.DOTALL)
        eval_list = json.loads(match.group(0)) if match else []
        return {item.get("id"): item for item in eval_list}
    except Exception as e:
        print(f"  ⚠️ Qwen parse notice: {e}")
        return {}


# Node 3: Deep Match Scoring
def evaluate_matches_node(state: PipelineState) -> Dict[str, Any]:
    print("\n⚖️ Node 3: Deep Match Scoring via Claude Pro...")
    raw_jobs = state.get("raw_jobs", [])
    if not raw_jobs:
        return {"evaluated_jobs": [], "low_match_summary": "No jobs found to score."}

    jobs_payload = "\n".join([
        f"ID {i+1}: {j.get('Job Profile')} at {j.get('Target Company')} ({j.get('Location')})"
        for i, j in enumerate(raw_jobs)
    ])

    eval_map = {}

    # 1. Primary: Direct Claude Pro Execution
    try:
        eval_map = call_claude_pro_evaluation(state['resume_summary'], jobs_payload)
        print("  ✓ Evaluation completed via Claude Pro (Direct CLI)!")
    except Exception as e:
        print(f"  ⚠️ Claude Pro CLI notice: {e}")
        # 2. Working Fallback: Local Qwen 2.5
        eval_map = evaluate_with_local_qwen(state['resume_summary'], jobs_payload)

    evaluated = []
    low_matches = []

    for i, job in enumerate(raw_jobs):
        ev = eval_map.get(i + 1, {})
        score = ev.get("match_score")
        
        # Safety heuristic if an ID was missed
        if score is None:
            title_lower = job.get("Job Profile", "").lower()
            score = 85 if any(k in title_lower for k in ["product", "chief of staff", "strategy", "builder", "lead"]) else 50

        job["Match Score"] = score
        job["Rating"] = ev.get("rating", "Good" if score >= 70 else "Average")
        job["Rationale"] = ev.get("rationale", "Aligned with core product leadership capabilities.")
        job["low_match_reason"] = ev.get("low_match_reason")

        if job["Match Score"] >= 60:
            evaluated.append(job)
        else:
            low_matches.append(job)

    evaluated.sort(key=lambda x: x.get("Match Score", 0), reverse=True)

    low_match_items = []
    for lm in low_matches[:4]:
        reason = lm.get("low_match_reason") or lm.get("Rationale")
        low_match_items.append(f"• {lm.get('Job Profile')} @ {lm.get('Target Company')} ({lm.get('Match Score')}%): {reason[:120]}")
    low_match_summary = "\n".join(low_match_items) if low_match_items else "No low matches."

    print(f"  ✓ {len(evaluated)} qualified positions approved for HITL sign-off.")
    return {"evaluated_jobs": evaluated, "low_match_summary": low_match_summary}


# Node 4: Batch Human-in-the-Loop (HITL) Sign-Off
def human_review_node(state: PipelineState) -> Dict[str, Any]:
    evaluated = state.get("evaluated_jobs", [])
    already_approved = state.get("approved_jobs", [])
    target = state.get("target_approved_count", 5)

    if not evaluated:
        print("\n⚠️ No new candidate jobs to review in this cycle.")
        return {"approved_jobs": already_approved, "user_decision": "proceed"}

    print("\n" + "="*68)
    print(f"🧑‍💼 HUMAN-IN-THE-LOOP SIGN-OFF (Approved so far: {len(already_approved)}/{target})")
    print("="*68)

    batch = evaluated[:5]
    remaining = evaluated[5:]

    for idx, job in enumerate(batch, 1):
        print(f"\n[{idx}] {job.get('Job Profile')} @ {job.get('Target Company')}")
        print(f"    Location: {job.get('Location')} | Match Score: {job.get('Match Score')}% ({job.get('Rating')})")
        print(f"    Rationale: {job.get('Rationale')}")
        print(f"    Link: {job.get('Job Link')}")

    print("\n" + "-"*68)
    user_input = input(
        f"Enter IDs to APPROVE (e.g., '1, 3'), 'all', 'none', or 'done' to stop: "
    ).strip().lower()

    newly_approved = []
    if user_input == "all":
        newly_approved.extend(batch)
    elif user_input not in ["none", "done", ""]:
        selected_indices = [int(i.strip()) for i in re.findall(r"\d+", user_input)]
        for i in selected_indices:
            if 1 <= i <= len(batch):
                newly_approved.append(batch[i - 1])

    total_approved = already_approved + newly_approved
    print(f"✓ Total Approved: {len(total_approved)}/{target}")

    if user_input == "done" or len(total_approved) >= target:
        decision = "proceed"
    elif remaining:
        decision = "review_more"
    else:
        fetch_more = input("Reviewed all jobs in this batch. Fetch broader live search? (y/n): ").strip().lower()
        decision = "search_more" if fetch_more == "y" else "proceed"

    return {
        "approved_jobs": total_approved,
        "evaluated_jobs": remaining,
        "user_decision": decision
    }


# Node 5: Real Contact Discovery (Exa SERP)





# ==========================================
# 2. UPDATED NODE 5 WITH GETEMAIL.IO ENRICHMENT
# ==========================================
import time

def verify_email_via_getemail(fullname: str, company: str, max_wait_seconds: int = 8) -> str:
    """Verifies live corporate email via GetEmail.io with automatic polling."""
    api_key = os.environ.get("GETEMAIL_API_KEY", "").strip()
    if not api_key or api_key == "your_actual_getemail_api_key_here":
        return None

    clean_domain = re.sub(r'[^a-zA-Z0-9]', '', company).lower()
    find_url = "https://api.getemail.io/v2/find-email"
    params = {
        "api_key": api_key,
        "fullname": fullname,
        "domain": clean_domain,
        "isEnrichToDo": "true"
    }

    try:
        resp = requests.get(find_url, params=params, timeout=12)
        if resp.status_code != 200:
            return None
            
        data = resp.json()
        if data.get("email"):
            return data.get("email")

        # Async polling for live SMTP handshake completion
        search_id = data.get("id")
        if data.get("code") == "EMAIL_FINDING_PROCESSING" and search_id:
            status_url = "https://api.getemail.io/v2/status-find-email"
            status_params = {"api_key": api_key, "id": search_id}
            
            start_time = time.time()
            while time.time() - start_time < max_wait_seconds:
                time.sleep(2.5)
                status_resp = requests.get(status_url, params=status_params, timeout=10)
                if status_resp.status_code == 200:
                    status_data = status_resp.json()
                    if status_data.get("email"):
                        return status_data.get("email")
                    if status_data.get("code") != "EMAIL_FINDING_PROCESSING":
                        break
    except Exception:
        pass

    return None


def find_contacts_node(state: PipelineState) -> Dict[str, Any]:
    print("\n🤝 Node 5: Enriching Approved Jobs with Real Recruiter Contacts & Verified Emails...")
    approved = state.get("approved_jobs", [])

    for job in approved:
        company = job.get("Target Company", "").strip()
        if not company or company in ["N/A", "Unknown Company", "Direct ATS"]:
            job["Prospect_Employer_Name"] = "Recruiting Team"
            job["Prospect_Employer_Title"] = "Talent Acquisition"
            job["Prospect_Employer_Profile_LI"] = "Not Found"
            job["Prospect_Employer_Profile_mail"] = "N/A"
            continue

        clean_comp = re.sub(r'(?i)\b(inc|ltd|pvt|technologies|corporation|llc|corp|co)\b', '', company).strip()
        clean_comp = re.sub(r'[^a-zA-Z0-9\s]', '', clean_comp).strip()
        domain_slug = re.sub(r'[^a-zA-Z0-9]', '', company).lower()

        # Step 1: Find real recruiter profile via Exa
        query = f'site:linkedin.com/in "{clean_comp or company}" ("Talent Acquisition" OR "Technical Recruiter" OR "Head of Product")'
        
        name = "Recruiting Team"
        headline = "Talent Acquisition"
        li_url = "Not Found"

        try:
            res = exa.search(
                query=query,
                type="auto",
                include_domains=["linkedin.com"],
                num_results=2
            )
            for r in res.results:
                if "/in/" in r.url:
                    li_url = r.url
                    raw_title = r.title.replace(" | LinkedIn", "").replace(" - LinkedIn", "")
                    parts = raw_title.split(" - ") if " - " in raw_title else raw_title.split(" | ")
                    name = parts[0].strip()
                    headline = parts[1].strip() if len(parts) > 1 else "Talent Acquisition"
                    break
        except Exception as e:
            print(f"  ⚠️ Recruiter search issue for {company}: {e}")

        # Step 2: Live SMTP Email Verification via GetEmail.io
        verified_email = None
        if name != "Recruiting Team":
            print(f"  📧 Verifying inbox for '{name}' at '{company}'...")
            verified_email = verify_email_via_getemail(fullname=name, company=clean_comp or company)

        # Step 3: Fallback formula if GetEmail has no match or credits are exhausted
        if not verified_email:
            first_clean = re.sub(r'[^a-zA-Z]', '', name.split()[0]).lower()
            if first_clean and len(first_clean) >= 3 and name != "Recruiting Team":
                verified_email = f"{first_clean}@{domain_slug}.com"
            else:
                verified_email = f"hiring@{domain_slug}.com"

        job["Prospect_Employer_Name"] = name
        job["Prospect_Employer_Title"] = headline
        job["Prospect_Employer_Profile_LI"] = li_url
        job["Prospect_Employer_Profile_mail"] = verified_email
        print(f"  ✓ {company}: {name} ({verified_email})")

    return {"approved_jobs": approved}

# Node 6: Deduplicating Excel Exporter
def export_excel_node(state: PipelineState) -> Dict[str, Any]:
    print("\n📊 Node 6: Appending Approved Jobs to Excel Database...")
    approved = state.get("approved_jobs", [])
    if not approved:
        print("  ⚠️ No approved jobs to export.")
        return {}

    new_rows = []
    for j in approved:
        new_rows.append({
            "Resume Applied With": state.get("candidate_name", "Default_CV"),
            "Job Profile": j.get("Job Profile", "N/A"),
            "Target Company": j.get("Target Company", "N/A"),
            "Location": j.get("Location", "Remote"),
            "Match Score": f"{j.get('Match Score', 'N/A')}%",
            "Rating": j.get("Rating", "N/A"),
            "Rationale": j.get("Rationale", "N/A"),
            "Job Link": j.get("Job Link", "N/A"),
            "Date Posted": j.get("Date Posted", state.get("duration_label", "Recent")),
            "Platform Name": j.get("Platform Name", "LinkedIn"),
            "Prospect_Employer_Name": j.get("Prospect_Employer_Name", "N/A"),
            "Prospect_Employer_Title": j.get("Prospect_Employer_Title", "N/A"),
            "Prospect_Employer_Profile_LI": j.get("Prospect_Employer_Profile_LI", "N/A"),
            "Prospect_Employer_Profile_mail": j.get("Prospect_Employer_Profile_mail", "N/A")
        })

    new_df = pd.DataFrame(new_rows)

    if os.path.exists(EXCEL_PATH):
        try:
            existing_df = pd.read_excel(EXCEL_PATH, sheet_name="Matched Jobs")
            final_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(
                subset=["Job Link"], keep="first"
            )
        except Exception:
            final_df = new_df
    else:
        final_df = new_df

    final_df.to_excel(EXCEL_PATH, index=False, sheet_name="Matched Jobs")
    print(f"  ✓ Successfully saved {len(final_df)} total entries in {EXCEL_PATH}")
    return {}

# ==========================================
# 6. GRAPH COMPILATION
# ==========================================
def hitl_routing(state: PipelineState) -> str:
    decision = state.get("user_decision", "proceed")
    if decision == "review_more":
        return "human_review"
    elif decision == "search_more":
        return "search_jobs"
    return "find_contacts"

workflow = StateGraph(PipelineState)

workflow.add_node("parse_resume", parse_resume_node)
workflow.add_node("search_jobs", search_jobs_node)
workflow.add_node("evaluate_matches", evaluate_matches_node)
workflow.add_node("human_review", human_review_node)
workflow.add_node("find_contacts", find_contacts_node)
workflow.add_node("export_excel", export_excel_node)

workflow.add_edge(START, "parse_resume")
workflow.add_edge("parse_resume", "search_jobs")
workflow.add_edge("search_jobs", "evaluate_matches")
workflow.add_edge("evaluate_matches", "human_review")

workflow.add_conditional_edges(
    "human_review",
    hitl_routing,
    {
        "human_review": "human_review",
        "search_jobs": "search_jobs",
        "find_contacts": "find_contacts"
    }
)

workflow.add_edge("find_contacts", "export_excel")
workflow.add_edge("export_excel", END)

app = workflow.compile()

# ==========================================
# 7. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*55)
    print("🤖 HITL JOB AGENT (Claude Pro + Exa Contacts + Local CV)")
    print("="*55 + "\n")

    available_cvs = []
    if os.path.exists(RESUMES_DIR):
        available_cvs = [os.path.splitext(f)[0] for f in os.listdir(RESUMES_DIR) if f.endswith(".pdf")]

    if available_cvs:
        print("Available Resumes in Directory:")
        for idx, cv in enumerate(available_cvs, 1):
            print(f"  [{idx}] {cv}")
        choice = input(f"\nSelect Persona (1-{len(available_cvs)}): ").strip()
        selected_persona = available_cvs[int(choice) - 1] if choice.isdigit() and 1 <= int(choice) <= len(available_cvs) else available_cvs[0]
    else:
        selected_persona = "Simar_Dhingra_CV"

    print(f"\n👉 Active Candidate Profile: {selected_persona}\n")

    dur_input = input("Enter posting recency (e.g. '24 hours', '3 days', '1 week') [Default: 1 week]: ").strip() or "1 week"
    loc_input = input("Enter locations (e.g. 'Delhi / Bangalore / Remote') [Default: Remote]: ").strip() or "Remote"
    roles_input = input("Enter target roles (e.g. 'Product Manager / Chief of Staff'): ").strip() or "Product Manager"
    target_count = input("How many approved jobs would you like to target? [Default: 5]: ").strip() or "5"

    inputs = {
        "candidate_name": selected_persona,
        "roles": [r.strip() for r in re.split(r"[/,]", roles_input) if r.strip()],
        "target_geos": [l.strip() for l in re.split(r"[/,]", loc_input) if l.strip()],
        "job_duration_days": parse_duration_to_days(dur_input),
        "duration_label": dur_input,
        "target_approved_count": int(target_count),
        "approved_jobs": [],
        "search_iteration": 1
    }

    print("\n🚀 Starting Pipeline Execution...\n")
    final_output = app.invoke(inputs)

    print("\n" + "="*55)
    print("📊 LOW MATCH JOBS ANALYSIS (<50 words per role)")
    print("="*55)
    print(final_output.get("low_match_summary"))
    print("\n✓ Process completed! Check Matched_Jobs.xlsx for your approved jobs and verified recruiter contacts.")

### job application agent

agent 1- a RAG agent which has access to my profile pdf, is able to extract relevant information from it like job titles, geo, years of experience, technical capability
Role- Senior Knowdlegdge management Specialist  
Goal- Converting the knowledge into legible, readible and comprehendible structured bullet points without changing any meaning or coherency.  
Backstory- You are a meticulous researcher. You specialize in parsing local documentation
        to answer queries without introducing outside hallucinations.

Task- to process the GIVEN PDF docs without editing anything and storing in memory

(I want to use multiple documents - here resumes)




agent 2 - a job researcher which is able to search for relevant jobs for {candidate_name} specific geo/profile/experience 
domain/industry etc.
Role- A specialist career job finder
Goal- to look for relevant jobs for the profiles provided  as per the user query: {user_query}
Backstory- You have successfully managed to search jobs from all known platforms which may have a job posting  (like LinkedIn/IIm jobs/wellfound, other social media platform posts, public forum posts by individuals and all local job posting sites for each geography provided)

Task- based on user query, find jobs posted online across different job listing platforms across organisations of all sizes posted at any duration
Output- A list of 25 jobs with their job description links 


agent3- a verifier which provides a similarity score b/w my profile doc and job description and background 
and verifies if the score is above a certain threshold
Role- Job Profile Evaluator in a Talent Acquisition Team
Goal- To search for relevancy of profile match 
Backstory- You have a background of matching candidate profiles for organisations ranging from new age startups to big tech and fortune 500 companies across globe. You also check if the job posting is updated, posted by a legitimate company/org

Task- Using the the links given, evaluate which job description title matches the agent 1 document. Cite a brief reason for match in less than 50 words
Output- a score out of Poor/Average/Good/Perfect match for profiles matching my resume. Cite a reason for match in less than 50 words 



agent 4- creates a list of jobs posted in the previous x duration 
Format should be excel with -
Role- Executive Assitant & Data Entry specialist
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
Backstory- You have history of reading and scraping useful information into a human readable structure & format. You are an expert at Excel 

Task- Convert the job matches with Average or higher score in an excel table to be accessed from system
Output-An excel table with the following columns:
1.JOb Profile- scrape the Title of the job
2.Job Link- the web link to directly access the job description
3. Platform Name- fetch/scrape the name of the platform where it is posted
4. Date posted- fetch the date of job post
Append all matched jobs to the above mentioned columns. 

Tools- excel


agent 5 - takes the input as csv and starts finding people on linkedIn who are senior to me or might be hiring in that org, anyone related to tech or management roles near to my role
Role- Networking Specialist
Goal- Headhunting best profiles via using web search
Backstory- You have a track record of hunting down profiles from all social media platforms who have or are likely to hire for relevant roles

Task- Based on the best profile matches, find 3 relevant profiles who are in either management roles or are senior in roles {candidate_name} is looking for
Output- A plan for cold reachout for the mentioned profiles



agent 6 -returns the best profile linkedin URLs + retrieves their email id using appolo as a tool, adds it in another column of the same excel
Role- data entry Specialist  
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
 
Backstory- You have a background in sales navigation and finding ways via official emails/ linkedIn profile links to reach out to prospect employers

Task-To retrieve emails and LinkedIn profiles for people selected by the agent 5
Output- add a column in the given excel with name "Prospect_Employer_Profile_mail" & "Prospect_Employer_Profile_LI" 
A list of 3 LinkedIn profile URLs & their official emails- each in a single cell 
Tools- LinkedIn_Search or google search or apollo.io search (whichever is accessible)




in London, Spain, Netherlands, Singapore, India or any Remote Roles
 